In [1]:
import zipfile
import struct
import pandas as pd
from datetime import datetime
import io

# ----------------------
# 1. 核心：从ZIP直接读取 .day 内存数据
# ----------------------
def read_tdx_day_from_zip(zip_path, day_filename_in_zip):
    """
    从 zip 压缩包中直接读取通达信 .day 文件，不解压到本地
    :param zip_path: zip 压缩包路径
    :param day_filename_in_zip: zip 里的文件名，如 sh/lday/sh563900.day
    :return: df
    """
    with zipfile.ZipFile(zip_path, 'r') as zf:
        # 直接在内存中打开文件
        with zf.open(day_filename_in_zip) as f:
            buffer = f.read()

    num_bars = len(buffer) // 32
    data = []

    for i in range(num_bars):
        bar = buffer[i*32 : (i+1)*32]
        date, open_, high, low, close, amount, vol, _ = struct.unpack("IIIIIfII", bar)

        # 日期
        dt = datetime(date//10000, (date%10000)//100, date%100)

        # 价格精度（自动识别 ETF / 股票）
        if "sh56" in day_filename_in_zip or "sh58" in day_filename_in_zip or "sz159" in day_filename_in_zip:
            open_ /= 1000
            high /= 1000
            low /= 1000
            close /= 1000
        else:
            open_ /= 100
            high /= 100
            low /= 100
            close /= 100

        data.append([dt, open_, high, low, close, vol, amount])

    df = pd.DataFrame(data, columns=["date", "open", "high", "low", "close", "volume", "amount"])
    return df

In [2]:
ZIP_FILE = r"D:\Users\lyx\Desktop\hsjday.zip" # 你的 zip 路径DAY_FILE = "sh/lday/sh563900.day" # 内部文件路径
DAY_FILE = "sh/lday/sz200b07.day" # 内部文件路径

In [3]:
df = read_tdx_day_from_zip(ZIP_FILE, DAY_FILE)
df

KeyError: "There is no item named 'sh/lday/sz200b07.day' in the archive"